# Student Performance Dataset - Data Dictionary and Data Cleaning

This notebook documents the Student Performance dataset, performs duplicate detection, analyzes missing values, applies data cleaning, and exports a cleaned version of the dataset.


## Dictionary

In [1]:
import pandas as pd

dictionary_data = [
    ["Hours_Studied", "Number of hours the student studied", "Numerical", "Hours", ">= 0", "Possible outliers or unrealistic values"],
    ["Attendance", "Attendance percentage of the student", "Numerical", "Percentage (%)", "0 to 100", "Values outside range or inconsistent decimals"],
    ["Parental_Involvement", "Level of parental involvement in education", "Categorical", "-", "Low / Medium / High", "Inconsistent labeling or typos"],
    ["Access_to_Resources", "Level of access to learning resources", "Categorical", "-", "Low / Medium / High", "Subjective categorization"],
    ["Extracurricular_Activities", "Participation in extracurricular activities", "Categorical", "-", "Yes / No", "Encoding required for modeling"],
    ["Sleep_Hours", "Average daily sleep hours", "Numerical", "Hours", "0 to 24", "Outliers or unrealistic values"],
    ["Previous_Scores", "Student's previous academic scores", "Numerical", "Score", "0 to 100", "Outliers or inconsistent scaling"],
    ["Motivation_Level", "Level of student motivation", "Categorical", "-", "Low / Medium / High", "Subjective and ordinal variable"],
    ["Internet_Access", "Whether the student has internet access", "Categorical", "-", "Yes / No", "Needs encoding"],
    ["Tutoring_Sessions", "Number of tutoring sessions attended", "Numerical", "Count", ">= 0", "Skewed distribution (many zeros)"],
    ["Final_Exam_Score", "Final exam score achieved by the student", "Numerical", "Score", "0 to 100", "Target variable; possible outliers"],
]

dictionary_columns = [
    "Name",
    "Description (Semantic Meaning)",
    "Data Type",
    "Units",
    "Possible Values / Categories",
    "Potential Data Quality Issues",
]

df_dictionary = pd.DataFrame(dictionary_data, columns=dictionary_columns)

pd.set_option("display.max_colwidth", None)
display(df_dictionary)

df_dictionary.to_csv("student_dictionary.csv", index=False)
print("Dictionary saved as: student_dictionary.csv")

,Name,Description (Semantic Meaning),Data Type,Units,Possible Values / Categories,Potential Data Quality Issues
0,Hours_Studied,Number of hours the student studied,Numerical,Hours,>= 0,Possible outliers or unrealistic values
1,Attendance,Attendance percentage of the student,Numerical,Percentage (%),0 to 100,Values outside range or inconsistent decimals
2,Parental_Involvement,Level of parental involvement in education,Categorical,-,Low / Medium / High,Inconsistent labeling or typos
3,Access_to_Resources,Level of access to learning resources,Categorical,-,Low / Medium / High,Subjective categorization
4,Extracurricular_Activities,Participation in extracurricular activities,Categorical,-,Yes / No,Encoding required for modeling
5,Sleep_Hours,Average daily sleep hours,Numerical,Hours,0 to 24,Outliers or unrealistic values
6,Previous_Scores,Student's previous academic scores,Numerical,Score,0 to 100,Outliers or inconsistent scaling
7,Motivation_Level,Level of student motivation,Categorical,-,Low / Medium / High,Subjective and ordinal variable
8,Internet_Access,Whether the student has internet access,Categorical,-,Yes / No,Needs encoding
9,Tutoring_Sessions,Number of tutoring sessions attended,Numerical,Count,>= 0,Skewed distribution (many zeros)


Dictionary saved as: student_dictionary.csv


## Load Student Performance Dataset

In [5]:
import os
import pandas as pd

dataset_file = "student_dataset.csv"

if not os.path.exists(dataset_file):
    raise FileNotFoundError(f"{dataset_file} not found")

df = pd.read_csv(dataset_file)

print(f"Loaded file: {dataset_file}")
print(f"Dataset shape before cleaning: {df.shape}")
display(df.head())

Loaded file: student_dataset.csv
Dataset shape before cleaning: (6607, 11)


,Hours_Studied,Attendance,Parental_Involvement,Access_to_Resources,Extracurricular_Activities,Sleep_Hours,Previous_Scores,Motivation_Level,Internet_Access,Tutoring_Sessions,Final_Exam_Score
0,23.5,84.2,Low,Low,Yes,7.4,50.3,Medium,Yes,1,54.3
1,19.7,63.6,Medium,High,Yes,6.7,83.1,Low,Yes,2,59.4
2,24.4,98.4,Medium,Medium,Yes,8.0,63.5,Medium,Yes,0,63.8
3,29.6,88.6,Medium,High,Yes,6.3,57.5,Medium,Yes,2,61.0
4,19.1,91.5,High,Medium,Yes,6.2,66.0,Medium,Yes,2,60.7


## Duplicate Detection and Handling

**Detection Strategy:**  
Since this dataset does not contain a unique identifier column, duplicate detection is based on full-row comparison. If all values in a row are repeated, that row is considered a duplicate.

**Consolidation Method:**  
If duplicate rows are found, the chosen method is removal while keeping the first occurrence. This avoids overrepresenting the same student record in future analyses.


In [6]:
# Identifying exact row duplicates
exact_duplicates = df.duplicated().sum()
print(f"Exact row duplicates: {exact_duplicates}")

# Removing duplicates
df_cleaned = df.drop_duplicates().copy()

print(f"Dataset after removing duplicates: {df_cleaned.shape}")

df_cleaned.to_csv("student_after_duplicate_removal.csv", index=False)
print("Intermediate file saved as: student_after_duplicate_removal.csv")

Exact row duplicates: 0
Dataset after removing duplicates: (6607, 11)
Intermediate file saved as: student_after_duplicate_removal.csv


## Missing Data Treatment

In [7]:
# Missing values per variable
missing_per_variable = df_cleaned.isnull().sum()
print("Missing values per variable:")
print(missing_per_variable[missing_per_variable > 0])

# Missing values overall
total_missing = df_cleaned.isnull().sum().sum()
print(f"\nTotal missing values in the dataset: {total_missing}")

Missing values per variable:
Series([], dtype: int64)

Total missing values in the dataset: 0


**Data Missing Patterns:**  
If missing values are found, they may occur either in numerical variables (for example, study hours or previous scores) or in categorical variables (for example, motivation level or internet access).

**Treatment Strategy and Justification:**  
- Numerical variables are filled with the **median**, since it is robust to outliers.  
- Categorical variables are filled with the **mode** (most frequent value), which preserves the most common category.  

This strategy preserves the largest possible number of records without introducing excessive distortion into the dataset.


In [8]:
# Separating numerical and categorical columns
numerical_cols = df_cleaned.select_dtypes(include=["number"]).columns
categorical_cols = df_cleaned.select_dtypes(exclude=["number"]).columns

# Filling missing numerical values with median
for col in numerical_cols:
    if df_cleaned[col].isnull().sum() > 0:
        df_cleaned[col] = df_cleaned[col].fillna(df_cleaned[col].median())

# Filling missing categorical values with mode
for col in categorical_cols:
    if df_cleaned[col].isnull().sum() > 0:
        df_cleaned[col] = df_cleaned[col].fillna(df_cleaned[col].mode()[0])

print(f"Total missing values after treatment: {df_cleaned.isnull().sum().sum()}")
print(f"\nDataset after cleaning: {df_cleaned.shape}")

df_cleaned.to_csv("student_performance_cleaned.csv", index=False)
print("Final cleaned file saved as: student_performance_cleaned.csv")

Total missing values after treatment: 0

Dataset after cleaning: (6607, 11)
Final cleaned file saved as: student_performance_cleaned.csv


## Optional Quality Checks

The following cell checks whether numerical values are within plausible ranges for this dataset.


In [9]:
quality_checks = {}

if "Attendance" in df_cleaned.columns:
    quality_checks["Attendance_out_of_range"] = ((df_cleaned["Attendance"] < 0) | (df_cleaned["Attendance"] > 100)).sum()

if "Sleep_Hours" in df_cleaned.columns:
    quality_checks["Sleep_Hours_out_of_range"] = ((df_cleaned["Sleep_Hours"] < 0) | (df_cleaned["Sleep_Hours"] > 24)).sum()

if "Previous_Scores" in df_cleaned.columns:
    quality_checks["Previous_Scores_out_of_range"] = ((df_cleaned["Previous_Scores"] < 0) | (df_cleaned["Previous_Scores"] > 100)).sum()

if "Final_Exam_Score" in df_cleaned.columns:
    quality_checks["Final_Exam_Score_out_of_range"] = ((df_cleaned["Final_Exam_Score"] < 0) | (df_cleaned["Final_Exam_Score"] > 100)).sum()

print("Quality checks:")
for key, value in quality_checks.items():
    print(f"{key}: {value}")

Quality checks:
Attendance_out_of_range: 0
Sleep_Hours_out_of_range: 0
Previous_Scores_out_of_range: 0
Final_Exam_Score_out_of_range: 0


## Final Statement

The dataset was preprocessed by documenting its variables, checking for duplicate rows, analyzing missing values, and applying cleaning strategies. Exact duplicates were removed when present. Missing numerical values were treated using median imputation, while missing categorical values were filled with the mode. The cleaned dataset was then exported for future analysis and modeling.
